In [2]:
import os  
import base64
from tqdm import tqdm
import time
from openai import AzureOpenAI  
from dotenv import load_dotenv
load_dotenv()

endpoint = os.getenv("AZURE_OPENAI_ENDPOINT_URL_2", "")  
deployment = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME_2", "")  
subscription_key = os.getenv("AZURE_OPENAI_API_KEY_2", "")  
api_version = os.getenv("AZURE_OPENAI_API_VERSION_2", "")  

# Initialize Azure OpenAI Service client with key-based authentication    
client = AzureOpenAI(  
    azure_endpoint=endpoint,  
    api_key=subscription_key,  
    api_version=api_version,
)

## Read validation data and label

In [3]:
import pandas as pd
validation_data= pd.read_csv("../type_classification-validation.csv")

In [4]:
#Prepare the chat prompt 
new_validation_df = pd.DataFrame(columns=["Sentence", "Result"])

print("Running labelling")
for index, row in tqdm(validation_data.iterrows(), total=len(validation_data)):
    chat_prompt = [{
        "role": "system",
        "content": [
            {
                "type": "text",
                "text": "You are an AI Model that help to classify whether a sentence can be used to support information to create Class Diagram or Use Case Diagram or Activity Diagram\n\nBelow are given 5 sentences and whether it is useful for any of the diagram\n1. \"AP : As a case progresses , I need to record all the individuals and organizations that Verdict: take part in the case activities and the specific role they play .\"\nUseful for Class Diagram, Useful for Use Case Diagram, Useful for Activity Diagram\n\n2. \"Unless you are a celebrity or a good friend of Romano you will need a reservation .\"\nVerdict: Useful for Class Diagram, Useful for Use Case Diagram, Not Useful for Activity Diagram\n\n3. \"Therefore , there can be overlapping table reservations .\"\nVerdict: Useful for Class Diagram, Not Useful for Use Case Diagram, Not Useful for Activity Diagram\n\n4. \"These samples are sometimes sub - divided and distributed to multiple research teams or labs for different specialized observations .\"\nVerdict: Not Useful for Class Diagram, Useful for Use Case Diagram, Useful for Activity Diagram\n\n5. \"When the reservation party arrives at Romano 's the reservation is assigned to one waiter .\"\nVerdict: Useful for Class Diagram, Useful for Use Case Diagram, Not Useful for Activity Diagram\n\nUser will put a sentence and decide whether it will be useful for any of the category as an example output like this [Useful Class, Useful Use Case, Not Useful Activity]\n\n"
            }
        ]
    }, {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": row['sentence']
            }
        ]
    }]
    
    if index % 30 == 0 and index != 0:
        time.sleep(60)
        
    # Generate the completion  
    completion = client.chat.completions.create(  
        model=deployment,
        messages=chat_prompt,
        temperature=0.7,  
        top_p=0.95,  
        frequency_penalty=0,  
        presence_penalty=0,
        stop=None,  
        stream=False
    )
    result = completion.choices[0].message.content
    new_validation_df = new_validation_df.append({"Sentence": row['sentence'], "Result": result}, ignore_index=True)
    if index % 10 == 0:
        print("Saved item: ", index+1)
        new_validation_df.to_csv("gpt-o3-label-validation-5 example.csv")
    
new_validation_df.to_csv("gpt-o3-label-validation-5 example.csv")

Running labelling


  0%|          | 0/145 [00:00<?, ?it/s]


BadRequestError: Error code: 400 - {'error': {'code': 'OperationNotSupported', 'message': 'The chatCompletion operation does not work with the specified model, o3-mini. Please choose different model and try again. You can learn more about which models can be used with each operation here: https://go.microsoft.com/fwlink/?linkid=2197993.'}}